## ❓ 1. Research Question (研究問題)
Before conducting the inferential analysis, we define our core statistical question:
* **Two-Sample Proportion Analysis (兩獨立樣本比例 Z 檢定):** "Is the proportion of current alcohol use different between male and female students?"

---

## 📊 2. Variable Definition and Data Check (變數定義與資料檢查)
In this section, we define the variables used and document the data cleaning process to ensure reproducibility:

### 2.1 Grouping Variable: `WhatIsYourSex`
* **Definition:** Biological sex of the high school student.
* **Recoding Rules (`Sex_Binary`):**
  * **0 (Failure):** Male (Originally coded as 2)
  * **1 (Success):** Female (Originally coded as 1)

### 2.2 Response Variable: `CurrentAlcoholUse`
* **Definition:** Measures whether students have consumed alcohol currently (within the past 30 days).
* **Recoding Rules (`Alcohol_Binary`):**
  * **0 (Failure):** No current use (Originally coded as 1)
  * **1 (Success):** Active user (Originally coded as 2 to 7)

### 2.3 Final Sample Size
* **Data Cleaning:** Missing or invalid values were handled using the `.dropna()` method. 
* **Final Analysis Sample Size:** $n = 12,659$ (Females: $n = 6,425$, Males: $n = 6,234$)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
import os

raw_filename = 'YRBS_2007.csv'

if os.path.exists(raw_filename):
    print(f"✅ 找到原始資料：{raw_filename}")
    raw_df = pd.read_csv(raw_filename)
    
    # 即時執行清理與重編碼邏輯（針對性別與飲酒變數）
    target_vars = ['WhatIsYourSex', 'CurrentAlcoholUse']
    df_analyzed = raw_df.dropna(subset=target_vars).copy()
    
    # 執行男女與飲酒的重編碼
    df_analyzed['Sex_Binary'] = df_analyzed['WhatIsYourSex'].apply(lambda x: 1 if x == 1 else (0 if x == 2 else np.nan))
    df_analyzed['Alcohol_Binary'] = df_analyzed['CurrentAlcoholUse'].apply(lambda x: 0 if x == 1 else (1 if 2 <= x <= 7 else np.nan))
    
    # 再次確保排除重編碼後產生的潛在缺失值
    df_analyzed = df_analyzed.dropna(subset=['Sex_Binary', 'Alcohol_Binary']).copy()
    print(f"✅ 資料處理完成，最終有效分析樣本數：{len(df_analyzed)}")
else:
    print(f"❌ 錯誤：在目前的資料夾找不到 {raw_filename}")
    print("請確認你已經把 CSV 檔拖進 Jupyter 與此 Notebook 同一目錄下。")
    raise FileNotFoundError

✅ 找到原始資料：YRBS_2007.csv
✅ 資料處理完成，最終有效分析樣本數：12659


---

## 📊 3. Descriptive Summary Table (描述性統計摘要)
Before running the hypothesis test, we aggregate the sample sizes, counts of active alcohol users, and sample proportions for each gender group.

In [6]:
# 建立描述性統計摘要表
summary_stats = df_analyzed.groupby('Sex_Binary')['Alcohol_Binary'].agg(['count', 'sum', 'mean'])
summary_stats.columns = ['Sample Size (n)', 'Current Alcohol Users (count)', 'Proportion (p_hat)']
summary_stats.index = ['Male (0)', 'Female (1)']

print("=== 描述性統計摘要表 ===")
print(summary_stats)

=== 描述性統計摘要表 ===
            Sample Size (n)  Current Alcohol Users (count)  Proportion (p_hat)
Male (0)               6234                           2853            0.457652
Female (1)             6425                           2864            0.445759


---

## 📈 4. Two-Sample Statistical Inference (兩樣本推論統計)
We conduct a **Two-Independent-Sample Z-Test for Proportions** to test if the difference between the two population proportions is statistically significant.

* **Hypotheses:**
  * $H_0: p_{\text{female}} - p_{\text{male}} = 0$ (There is no difference in alcohol use proportions between genders)
  * $H_a: p_{\text{female}} - p_{\text{male}} \neq 0$ (There is a significant difference between genders)

In [11]:
# 提取兩組數據
n_male = summary_stats.loc['Male (0)', 'Sample Size (n)']
success_male = summary_stats.loc['Male (0)', 'Current Alcohol Users (count)']
p_hat_male = summary_stats.loc['Male (0)', 'Proportion (p_hat)']

n_female = summary_stats.loc['Female (1)', 'Sample Size (n)']
success_female = summary_stats.loc['Female (1)', 'Current Alcohol Users (count)']
p_hat_female = summary_stats.loc['Female (1)', 'Proportion (p_hat)']

counts = np.array([success_female, success_male])
nobs = np.array([n_female, n_male])

# 1. 執行雙尾 z 檢定
z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

# 2. 計算男女比例差異的 95% 信賴區間 (Wald Method)
ci_low, ci_upp = confint_proportions_2indep(success_female, n_female, success_male, n_male, alpha=0.05, method='wald')
prop_diff = p_hat_female - p_hat_male

print("=== 兩樣本推論結果 ===")
print(f"Point Estimate of Difference (Female - Male): {prop_diff:.4f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f} (or {p_value:.4e})")
print(f"95% Confidence Interval for Diff: ({ci_low:.4f}, {ci_upp:.4f})\n")

=== 兩樣本推論結果 ===
Point Estimate of Difference (Female - Male): -0.0119
Z-statistic: -1.3442
P-value: 0.1789 (or 1.7887e-01)
95% Confidence Interval for Diff: (-0.0292, 0.0054)



---

## 📝 5. Statistical Interpretation (結果解釋)

* **Descriptive Summary (描述性統計)：**
  在有效樣本高中生中，女性的目前飲酒比例為 **44.6%**，男性的目前飲酒比例為 **45.8%**。樣本中觀察到女性的飲酒比例略低於男性約 1.19%（點估計差異 = -0.0119）。

* **Z-Test & P-value (假設檢定)：**
  雙樣本比例 Z 檢定結果顯示，檢定統計量 $Z = -1.3442$，雙尾 **$P\text{-value} = 0.1789$**。由於 $P\text{-value} > 0.05$，在 $\alpha = 0.05$ 的顯著水準下，我們**無法拒絕虛無假設（Fail to reject $H_0$）**。這代表男女學生之間的飲酒比例在統計上沒有顯著差異。

* **Confidence Interval (信賴區間)：**
  男女飲酒比例差異的 95% 信賴區間為 **$[-0.0292, 0.0054]$**。因為**該信賴區間包含了 0**，這說明兩群體的真實母體比例完全相同的可能性非常高，進一步驗證了檢定不顯著的結論。

* **Final Conclusion (最終結論)：**
  我們沒有足夠的證據支持高中男女學生的目前飲酒率存在根本性差異。樣本中觀察到的微小差距（1.19%）純粹屬於隨機抽樣造成的誤差（Sampling error）。

In [23]:
# ==========================================
# 6. 建立符合正式報告格式的綜合統計摘要表並自動匯出
# ==========================================

# 1. 為了完美符合你的表格格式，我們手動抽取出對應數值
# 男性 (0) 與 女性 (1) 的個別資料
n_m, count_m, p_m = int(n_male), int(success_male), p_hat_male
n_f, count_f, p_f = int(n_female), int(success_female), p_hat_female

# 2. 建構符合圖片欄位結構的字典
extended_summary_data = {
    "變數名稱 (Variable)": [
        "Alcohol_Binary (目前飲酒行為)", 
        "Alcohol_Binary (目前飲酒行為)"
    ],
    "資料型態 (Type)": [
        "類別數值", 
        "類別數值"
    ],
    "類別 (Category)": [
        "Male (0)", 
        "Female (1)"
    ],
    "有效樣本/人數 (Count)": [
        n_m, 
        n_f
    ],
    "平均數/比例 (Mean/Prop)": [
        f"{p_m * 100:.2f}%", 
        f"{p_f * 100:.2f}%"
    ],
    "標準差 (Std. Dev)": [
        "-", 
        "-"
    ],
    "95% 置信區間 (95% CI)": [
        "-",  # 單獨群體通常不放差異信賴區間，或可在下方備註
        f"({ci_low:.4f}, {ci_upp:.4f})*" # 將男女差異信賴區間標記於此
    ],
    "檢定假說 (H0)": [
        "-", 
        "p_female = p_male"
    ],
    "p-value": [
        "-", 
        f"{p_value:.4f}" if p_value >= 0.001 else "< 0.001"
    ]
}

# 3. 轉換為 DataFrame
df_summary_custom = pd.DataFrame(extended_summary_data)

# 定義輸出的檔名
output_file = 'inference_summary_table.csv'

# 匯出至 CSV (包含 utf-8-sig 防止 Excel 亂碼)
df_summary_custom.to_csv(output_file, index=False, encoding='utf-8-sig')

print("=== 綜合統計摘要表 (正式報告格式) ===")
display(df_summary_custom)
print(f"\n💾 成功！符合新格式的摘要表已儲存至：{output_file}")
print("💡 註：在 95% 置信區間欄位中，標記 * 處為兩樣本『比例差異』的 95% 信賴區間。")

=== 綜合統計摘要表 (正式報告格式) ===


,變數名稱 (Variable),資料型態 (Type),類別 (Category),有效樣本/人數 (Count),平均數/比例 (Mean/Prop),標準差 (Std. Dev),95% 置信區間 (95% CI),檢定假說 (H0),p-value
0,Alcohol_Binary (目前飲酒行為),類別數值,Male (0),6234,45.77%,-,-,-,-
1,Alcohol_Binary (目前飲酒行為),類別數值,Female (1),6425,44.58%,-,"(-0.0292, 0.0054)*",p_female = p_male,0.1789



💾 成功！符合新格式的摘要表已儲存至：inference_summary_table.csv
💡 註：在 95% 置信區間欄位中，標記 * 處為兩樣本『比例差異』的 95% 信賴區間。
